# 골드셋(정답지) 구축 — Colab

프로필별 **정답 논문 라벨(0/1)**을 만드는 데 필요한 기능만 모은 노트북.
(검색·수집·삭제 같은 일반 기능은 `colab_client.ipynb` 참고)

## 워크플로우
1. 접속(1) · 공통 함수(2) · 프로필 로드(3) 실행
2. **(4) Gemini 프로필 처리** — 한국어 프로필을 영어로 번역 + 키워드 추출
3. **(5) 키워드 DF 검증** — 키워드가 너무 흔/희귀하지 않은지 (50~500)
4. **(6) 라벨링 후보 CSV** — 프로필당 30편 뽑아 다운로드
5. **(7) 라벨링 가이드** 보고 CSV의 label(0/1)·tag 채우기
6. **(8) 라벨 업로드** — 채운 CSV를 서버에 반영
7. **(9) 현황·agreement** — judge vs 사람 일치율(κ) 확인

> 서버가 최신 코드로 배포돼 있어야 합니다 (EC2: `git pull` → 재시작 → `python migrate_fts.py`).
> API 키는 실행 시 입력창으로 받아 노트북에 저장되지 않습니다.

## 1. 접속 정보 설정

In [ ]:
from getpass import getpass

API_BASE = None; API_KEY = None
try:
    from google.colab import userdata
    try:
        API_BASE = userdata.get("ARXIV_API_BASE"); API_KEY = userdata.get("ARXIV_API_KEY")
        print("보안 비밀에서 접속 정보 로드")
    except Exception: pass
except ImportError: pass
if not API_BASE: API_BASE = input("API_BASE (예: http://1.2.3.4:8000): ").strip()
if not API_KEY: API_KEY = getpass("API_KEY: ").strip()
API_BASE = API_BASE.rstrip("/"); HEADERS = {"X-API-Key": API_KEY}
print("서버:", API_BASE)

## 2. 공통 함수 (골드셋용)

In [ ]:
import requests
import pandas as pd

def _post(path, payload, timeout=120):
    r = requests.post(f"{API_BASE}{path}", json=payload, headers=HEADERS, timeout=timeout); r.raise_for_status(); return r.json()
def _get(path, timeout=30, **params):
    r = requests.get(f"{API_BASE}{path}", params=params, headers=HEADERS, timeout=timeout); r.raise_for_status(); return r.json()

def health():
    r = requests.get(f"{API_BASE}/health", timeout=10); r.raise_for_status(); return r.json()

def keyword_df(term, category=None):
    """키워드 DF(문서빈도) — 50~500 검증용."""
    return _get("/keyword_df", term=term, category=category)

def labeling_pool(profile_text, keywords, category=None, total=30, n_random=0, n_keyword=15, m_embedding=70, recent_days=365):
    """골드셋 라벨링 후보 = 하이브리드(키워드+임베딩) 상위 total편(기본 30 = 키워드15+임베딩15). 최근 recent_days일(기본 1년) 이내 논문만."""
    return _post("/labeling_pool", {"profile_text": profile_text, "keywords": keywords, "category": category,
                                    "total": total, "n_random": n_random, "n_keyword": n_keyword,
                                    "m_embedding": m_embedding, "recent_days": recent_days})

def upload_labels(labels):
    """라벨 업로드. labels=[{profile_id, arxiv_id, labeler, label, source?, tag?}]"""
    return _post("/labels", {"labels": labels})

def get_labels(profile_id=None):
    return _get("/labels", profile_id=profile_id)

def get_paper(arxiv_id):
    return _get(f"/papers/{arxiv_id}")

print("공통 함수 정의 완료 — 접속 확인:", health())

## 3. 프로필 로드 (12개, 내장)
셀 실행만으로 로드. 프로필 수정은 이 JSON을 직접 편집 (repo `profiles.json`과 동기화 권장).

In [ ]:
import json
from collections import Counter

PROFILES = json.loads('''
[
  {
    "profile_id": "P1",
    "category": "cs.RO",
    "title": "Grasping",
    "profile_text": "I am interested in grasping within robot manipulation. I especially care about stable grasping using tactile sensors and force control, and about implementing human-hand grasping mechanisms on robots. I prefer optimization-based methods (contact mechanics, force closure, optimization approaches to grasp planning) over learning-based approaches.",
    "profile_text_ko": "로봇 매니퓰레이션 중에서도 grasping(파지)에 관심이 있다. 특히 tactile sensor와 force control을 이용해 로봇이 물체를 안정적으로 잡는 방법과, 사람 손의 grasping 메커니즘을 로봇에 구현하는 방식에 흥미가 있다. Learning-based 접근보다는 optimization 기반 방법론(contact mechanics, force closure, grasp planning의 최적화적 접근)을 선호한다.",
    "keywords": ["tactile sensing", "force control", "grasp planning", "dexterous manipulation"],
    "keywords_confirmed": true,
    "exclusion": "learning-based/RL 기반 grasping, navigation/SLAM 등 grasping과 무관한 로봇 주제, grasping 외 매니퓰레이션 하위분야"
  },
  {
    "profile_id": "P2",
    "category": "cs.RO",
    "title": "VLA",
    "profile_text": "I am interested in Vision-Language-Action (VLA) models, especially how VLA connects to manipulation tasks and how actions are represented and processed. Rather than the VLA architecture or training method itself, I care more about which real manipulation tasks VLA can be applied to.",
    "profile_text_ko": "Vision-Language-Action(VLA) 모델에 관심이 있다. 특히 VLA가 manipulation task에 어떻게 연결되는지, action을 어떤 방식으로 표현·처리하는지가 궁금하다. VLA 자체의 아키텍처·학습 방법보다는, VLA를 실제 어떤 manipulation task에 적용할 수 있는지에 더 관심이 있다.",
    "keywords": ["vision-language-action model", "language-conditioned manipulation", "action representation", "robot manipulation policy"],
    "keywords_confirmed": false,
    "exclusion": "VLA 아키텍처·학습 방법 자체만 다루고 매니퓰레이션 응용이 없는 연구"
  },
  {
    "profile_id": "P3",
    "category": "cs.RO",
    "title": "Off-road Navigation",
    "profile_text": "I am interested in off-road navigation. Compared to indoor navigation, I am curious how robots perceive and reason about unstructured outdoor environments, and whether the perception-processing pipeline itself differs between indoor and outdoor settings.",
    "profile_text_ko": "Off-road navigation에 관심이 있다. 실내 navigation과 비교해서, 로봇이 비정형 외부 환경을 어떻게 인지하고 판단하는지 그 메커니즘의 차이가 궁금하다. 환경 자체의 물리적 특성 차이뿐 아니라, 인식한 데이터가 처리되는 파이프라인 자체에서도 실내/실외 간 차이가 있는지에 관심이 있다.",
    "keywords": ["off-road navigation", "terrain perception", "unstructured environment", "outdoor navigation"],
    "keywords_confirmed": false,
    "exclusion": "실내(indoor) 전용 navigation 연구"
  },
  {
    "profile_id": "P4",
    "category": "cs.RO",
    "title": "3D 인식 → 매니퓰레이션 액션 플래닝",
    "profile_text": "I am interested in how robots perceive 3D objects and how that perception is converted into representations for manipulation. In particular, how perception data such as point clouds and 6D pose is passed to downstream action planning, and the representation and processing pipeline involved.",
    "profile_text_ko": "로봇이 3D 물체를 인식하는 과정과, 그 인식 결과가 매니퓰레이션을 위한 데이터로 어떻게 변환·표현되는지에 관심이 있다. 특히 포인트클라우드, 6D pose 같은 인식 데이터가 이후 action planning에 어떤 형태로 전달되는지, 그 표현 방식과 처리 파이프라인에 흥미가 있다.",
    "keywords": ["6D pose estimation", "point cloud", "manipulation action planning", "3D object perception"],
    "keywords_confirmed": false,
    "exclusion": "인식 결과가 매니퓰레이션으로 이어지지 않는 순수 3D 인식 연구"
  },
  {
    "profile_id": "P5",
    "category": "cs.CV",
    "title": "자율주행 3D 인지",
    "profile_text": "I research computer vision for autonomous driving. I am interested in 3D object detection, BEV (Bird's Eye View), and multi-camera perception. I want to exclude LiDAR-only research and path/trajectory planning research.",
    "profile_text_ko": "자율주행을 위한 컴퓨터 비전 기술을 연구하고 있다. 3D Object Detection, BEV(Bird's Eye View), Multi-Camera Perception 관련 연구에 관심이 있다. LiDAR만을 사용하는 연구나 경로 계획(Path Planning) 연구는 제외하고 싶다.",
    "keywords": ["3D object detection", "bird's eye view", "multi-camera perception", "camera-based 3D detection", "autonomous driving perception"],
    "keywords_confirmed": true,
    "exclusion": "LiDAR-only 연구, Path Planning 및 Trajectory Planning 연구"
  },
  {
    "profile_id": "P6",
    "category": "cs.CV",
    "title": "Medical Foundation Models",
    "profile_text": "I research medical image analysis, focused on applying foundation models (SAM, MedSAM, DINO, MAE) to medical image segmentation and detection. I prefer research using prompt learning, adapters, parameter-efficient fine-tuning (PEFT), and vision-language models for medical imaging. I want general natural-image foundation model research ranked lower.",
    "profile_text_ko": "의료 영상 분석을 연구하고 있으며 Foundation Model(SAM, MedSAM, DINO, MAE 등)을 의료 영상 분할 및 검출에 적용하는 연구에 관심이 있다. Prompt Learning, Adapter, Parameter-Efficient Fine-tuning(PEFT), Vision-Language Model을 활용한 의료 영상 연구를 선호한다. 일반 자연영상 Foundation Model 연구는 우선순위를 낮게 추천받고 싶다.",
    "keywords": ["medical image segmentation", "foundation model", "parameter-efficient fine-tuning", "SAM medical imaging"],
    "keywords_confirmed": false,
    "exclusion": "일반 자연영상 Foundation Model 연구 (우선순위 낮게)"
  },
  {
    "profile_id": "P7",
    "category": "cs.CV",
    "title": "Diffusion 이미지 생성·편집",
    "profile_text": "I research image generation. I am more interested in diffusion models than GANs, and want image editing and inpainting papers prioritized. I exclude research that deals only with text generation.",
    "profile_text_ko": "이미지 생성 분야를 연구하고 있다. GAN보다 Diffusion Model을 이용한 이미지 생성 연구에 더 관심이 있으며, 이미지 편집과 Inpainting 관련 최신 논문를 우선적으로 추천받고 싶다. 텍스트 생성만을 다루는 연구는 제외한다.",
    "keywords": ["diffusion model", "image editing", "image inpainting", "image generation"],
    "keywords_confirmed": false,
    "exclusion": "텍스트 생성만 다루는 연구, GAN 전용 기법"
  },
  {
    "profile_id": "P8",
    "category": "cs.CV",
    "title": "Test-Time Adaptation",
    "profile_text": "I research computer vision, especially test-time adaptation (TTA) and domain generalization. I want methods that adapt a model at test time without additional training data, and online adaptation research, prioritized.",
    "profile_text_ko": "Computer Vision 분야를 연구하고 있다. 특히 Test-Time Adaptation(TTA)과 Domain Generalization 연구에 관심이 많다. 추가 학습 데이터 없이 테스트 환경에서 모델을 적응시키는 방법과 Online Adaptation 연구를 우선적으로 추천받고 싶다.",
    "keywords": ["test-time adaptation", "domain generalization", "online adaptation", "distribution shift"],
    "keywords_confirmed": false,
    "exclusion": "테스트 시점 적응/일반화와 무관한 일반 학습 연구"
  },
  {
    "profile_id": "P9",
    "category": "cs.CL",
    "title": "LLM 추론 최적화",
    "profile_text": "I am interested in LLM inference efficiency, especially techniques that serve an already-trained model cheaper and faster, such as speculative decoding, KV cache compression, and post-training quantization. Rather than pretraining scaling laws or training-time efficiency, I focus on inference-time optimization of deployed models.",
    "profile_text_ko": "LLM 추론(inference) 효율화에 관심이 있다. 특히 speculative decoding, KV cache 압축, post-training quantization처럼 이미 학습이 끝난 모델을 더 싸고 빠르게 서빙하는 기법에 흥미가 있다. 사전학습 스케일링 법칙이나 학습(training) 단계의 효율화보다는, 배포된 모델의 추론 시점 최적화에 집중하고 싶다.",
    "keywords": ["speculative decoding", "KV cache", "post-training quantization", "LLM serving"],
    "keywords_confirmed": true,
    "exclusion": "사전학습 스케일링/학습 단계 효율화, 추론 가속용 하드웨어(가속기) 설계 자체, 효율화와 무관한 LLM 능력 평가"
  },
  {
    "profile_id": "P10",
    "category": "cs.CL",
    "title": "RAG",
    "profile_text": "I am interested in retrieval-augmented generation, especially how retrieval quality affects the factuality of generated answers, and citation grounding (identifying which source documents an answer came from). For hallucination detection and mitigation I do not mind whether the approach uses internal model signals or external verification. Pure IR ranking research without any generation component is outside my interest.",
    "profile_text_ko": "Retrieval-augmented generation에 관심이 있다. 특히 검색 품질이 생성 결과의 사실성(factuality)에 어떻게 영향을 주는지, 그리고 생성된 답이 어떤 근거 문서에서 왔는지를 밝히는 citation grounding이 궁금하다. 환각(hallucination)을 탐지하고 줄이는 방법이라면 모델 내부 신호를 쓰든 외부 검증을 쓰든 접근 방식을 가리지 않는다. 다만 생성 요소 없이 검색 랭킹만 다루는 순수 IR 연구는 관심 밖이다.",
    "keywords": ["retrieval-augmented generation", "citation grounding", "hallucination detection", "factuality"],
    "keywords_confirmed": false,
    "exclusion": "생성 요소가 없는 순수 IR 랭킹 연구"
  },
  {
    "profile_id": "P11",
    "category": "cs.CL",
    "title": "LLM 에이전트",
    "profile_text": "I am interested in LLM agents that use external tools: how reliably function calling and tool use work in real environments, and how role division and multi-step planning happen when multiple agents collaborate. I focus on agents in software/API environments rather than game-playing RL agents or embodied robotic agents.",
    "profile_text_ko": "외부 도구를 사용하는 LLM 에이전트에 관심이 있다. function calling과 tool use가 실제 환경에서 얼마나 신뢰할 수 있게 동작하는지, 그리고 여러 에이전트가 협업할 때 역할 분담과 다단계 계획(planning)이 어떻게 이루어지는지가 궁금하다. 게임을 플레이하는 RL 에이전트나 로봇에 탑재되는 embodied 에이전트보다는, 소프트웨어/API 환경에서 동작하는 에이전트에 집중하고 싶다.",
    "keywords": ["tool use", "function calling", "LLM agent", "multi-agent collaboration"],
    "keywords_confirmed": false,
    "exclusion": "게임 RL 에이전트, embodied 로봇 에이전트 (소프트웨어/API 환경만)"
  },
  {
    "profile_id": "P12",
    "category": "cs.CL",
    "title": "LLM 평가",
    "profile_text": "I am interested in how to evaluate LLM systems: how far LLM-as-a-judge can replace human evaluation, judge biases (position bias, self-preference) and reliability, and benchmark design and contamination detection. I prefer research that tackles the evaluation problem itself over papers that only report benchmark scores without a new evaluation methodology.",
    "profile_text_ko": "LLM 시스템을 어떻게 평가할 것인가 자체에 관심이 있다. 특히 LLM-as-a-judge가 사람 평가를 어디까지 대체할 수 있는지, judge가 갖는 편향(위치 편향, 자기 선호 등)과 신뢰성 문제가 궁금하다. 벤치마크 설계와 오염(contamination) 탐지에도 흥미가 있다. 새로운 평가 방법론 제안 없이 기존 벤치마크 점수만 보고하는 논문보다는, 평가라는 문제 그 자체를 다루는 연구를 보고 싶다.",
    "keywords": ["LLM-as-a-judge", "benchmark contamination", "evaluation methodology", "human preference evaluation"],
    "keywords_confirmed": false,
    "exclusion": "새로운 평가 방법론 없이 기존 벤치마크 점수만 보고하는 논문"
  }
]
''')

print(len(PROFILES), "개 프로필  ", dict(Counter(p["category"] for p in PROFILES)))
print("키워드 확정(사람):", [p["profile_id"] for p in PROFILES if p.get("keywords_confirmed")])

## 4. Gemini 프로필 처리 (한국어 → 영어 번역 + 키워드 추출)
`profile_text_ko`를 Gemini 2.5 Flash에 보내 영어 본문·키워드·제외조건을 추출.
**사람이 확정한 키워드(P1/P5/P9)는 보존**하고 번역만 갱신.

In [ ]:
%pip install -q google-genai

In [ ]:
from getpass import getpass
from google import genai
from google.genai import types

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    try: GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception: pass
except ImportError: pass
if not GEMINI_API_KEY: GEMINI_API_KEY = getpass("GEMINI_API_KEY: ").strip()

gclient = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-2.5-flash"
PROMPT = """You are helping build a research-paper recommender over English arXiv abstracts.
Given a researcher's interest profile written in Korean, produce:
1. "profile_text": a faithful, concise English translation, used as an embedding search query.
2. "keywords": 3-5 English keywords/phrases for BM25 search. Prefer specific 2-3 word technical phrases over broad field names.
3. "exclusion": one short English phrase for what to filter out. Empty string if none.
Return ONLY valid JSON with keys: profile_text, keywords (array), exclusion.

Korean profile:
{body}
"""

def gemini_extract(body):
    r = gclient.models.generate_content(model=GEMINI_MODEL, contents=PROMPT.format(body=body),
        config=types.GenerateContentConfig(response_mime_type="application/json"))
    t = (r.text or "").strip()
    if t.startswith("```"):
        t = t.strip("`"); t = t[t.find("{"): t.rfind("}") + 1]
    return json.loads(t)

for p in PROFILES:
    try:
        out = gemini_extract(p.get("profile_text_ko") or p["profile_text"])
    except Exception as e:
        print(f"[오류] {p['profile_id']}: {e}"); continue
    if out.get("profile_text"): p["profile_text"] = out["profile_text"].strip()
    if out.get("keywords") and not p.get("keywords_confirmed", False):
        p["keywords"] = [k.strip() for k in out["keywords"] if k.strip()]
    if out.get("exclusion"): p["exclusion_en"] = out["exclusion"].strip()
    print(f"{p['profile_id']}: {p['keywords']}")
print("\n프로필 처리 완료 → 5번 DF 검증으로 확인")

## 5. 키워드 DF 검증 (전체 프로필)
각 키워드가 서버 DB에서 몇 편 매칭되는지 세어 50~500 구간을 벗어나는 것 표시.
벗어나면 4번(Gemini)이나 수동으로 키워드를 조정. (임계값은 DB 규모에 맞게 조정 가능)

In [ ]:
LO, HI = 50, 500
flagged = 0
for p in PROFILES:
    print(f"\n=== {p['profile_id']} [{p['category']}] ===")
    for kw in p["keywords"]:
        d = keyword_df(kw, category=p["category"]); n = d["df"]
        flag = "OK" if LO <= n <= HI else ("TOO_COMMON" if n > HI else "TOO_RARE")
        if flag != "OK": flagged += 1
        print(f"  [{flag:11s}] DF={n:5d} (cat={d['df_in_category']})  {kw}")
print(f"\n조정 필요: {flagged}개 (구간 {LO}~{HI})")

## 6. 라벨링 후보 CSV (프로필당 30편)
하이브리드(키워드+임베딩) 30편(키워드15+임베딩15)을 프로필마다 뽑아 CSV로 저장, zip 다운로드.
**후보는 최근 1년(365일) 이내 논문만** — 3년~1년 전 논문은 keyword_df 계산에만 쓰임.
`source`: keyword/embedding/both. `label`/`tag`는 빈 칸 → 7번 가이드 보고 채움.

In [ ]:
import os, shutil

os.makedirs("labeling", exist_ok=True)
COLS = ["arxiv_id", "source", "label", "tag", "title", "primary_category", "submitted_date", "abstract_clean"]
for p in PROFILES:
    pool = labeling_pool(p["profile_text"], p["keywords"], category=p["category"], total=30, n_random=0)
    df = pd.DataFrame(pool); df["label"] = ""; df["tag"] = ""
    df[[c for c in COLS if c in df.columns]].to_csv(f"labeling/{p['profile_id']}.csv", index=False)
    print(f"{p['profile_id']}: {len(pool)}편")
shutil.make_archive("labeling", "zip", "labeling")
print("\nlabeling.zip 생성")
from google.colab import files; files.download("labeling.zip")

## 7. 라벨링 가이드 (0/1 + uncertain)

각 후보를 **제목+초록만** 보고 판단 (PDF 열지 않기, 한 편 30초 내). 기준은 "내 프로필과의 관련성"(논문 유명세 아님).

**단계별 판단**
1. **제외조건에 명시적으로 걸리는가?** → 걸리면 무조건 `label=0` (여기서 끝, 애매함 없음)
2. **이 논문의 핵심 기여/실험이 프로필 포함조건과 직접 관련 있는가?**
   - 핵심 기여면 → `label=1`
   - 관련 키워드가 서론/관련연구에서만 언급되고 실제 기여는 다른 것이면 → `label=0`
3. **그래도 애매하면**: "제목만 보고 프로필 담당자에게 보여주면 읽어볼 것 같은가?"
   - 그래도 애매하면 `tag=uncertain` 달고 넘어가 → 나중에 팀 논의로 확정

**태그(선택)**: `uncertain`(애매), `kw_only`(키워드만 맞고 무관), `excl_violation`(제외조건 위반)

**예시**
- 키워드 `tactile sensor`가 있어도 논문 기여가 센서 하드웨어 설계이고 grasping은 도입부에서만 언급 → 프로필1(grasping) 입장 `0`
- `RL 기반 grasping` 논문 → 프로필1은 제외조건(learning-based)에 명시적으로 걸리니 1단계에서 `0`. 프로필2(VLA)엔 애초에 무관 `0`

## 8. 라벨 업로드
라벨을 채운 CSV를 다시 올려(왼쪽 파일탭) 서버에 반영. `labeler`는 본인 이름(judge 채점이면 'judge').

In [ ]:
LABELER = input("라벨러 이름 (judge 또는 본인 이름): ").strip()
PROFILE_ID = input("업로드할 프로필 id (예: P1): ").strip()

# 채운 CSV 읽기 (Colab 파일탭에 labeling/P1.csv 를 올려둔 상태)
done = pd.read_csv(f"labeling/{PROFILE_ID}.csv")
done = done[done["label"].astype(str).str.strip() != ""]
labels = []
for _, r in done.iterrows():
    tag = r.get("tag")
    labels.append({"profile_id": PROFILE_ID, "arxiv_id": r["arxiv_id"], "labeler": LABELER,
                   "label": int(float(r["label"])), "source": r.get("source"),
                   "tag": (tag if isinstance(tag, str) and tag.strip() else None)})
print(f"업로드 {len(labels)}건...")
print(upload_labels(labels))

## 9. 라벨 현황 · judge vs 사람 일치율(κ)
전체 라벨 현황과, 같은 논문에 judge·사람 라벨이 둘 다 있는 경우의 일치율/Cohen κ를 계산.
(κ가 낮으면 라벨 기준을 다시 맞추거나 judge 프롬프트 수정)

In [ ]:
labels = get_labels()
df = pd.DataFrame(labels)
print("총 라벨:", len(df))
if not df.empty:
    print("\n라벨러별 건수:"); print(df.groupby("labeler").size())
    piv = df.pivot_table(index=["profile_id", "arxiv_id"], columns="labeler", values="label", aggfunc="first")
    if "judge" in piv.columns:
        for h in [c for c in piv.columns if c != "judge"]:
            pair = piv[["judge", h]].dropna()
            if len(pair):
                agree = (pair["judge"] == pair[h]).mean()
                try:
                    from sklearn.metrics import cohen_kappa_score
                    k = cohen_kappa_score(pair["judge"], pair[h])
                    print(f"judge vs {h}: n={len(pair)}  일치율={agree:.2f}  κ={k:.2f}")
                except Exception:
                    print(f"judge vs {h}: n={len(pair)}  일치율={agree:.2f}")
    else:
        print("\n(judge 라벨이 아직 없어 일치율 계산 불가)")